# Multi-Vector ANN: Indexing and Pruning MaxSim at Scale (PLAID)

`late-interaction-learned-sparse` lifted the single-vector rank ceiling by keeping one contextual
vector per **token** and scoring by MaxSim, $S(q,d)=\sum_i\max_j\langle q_i,d_j\rangle$, and flagged the
bill: an index roughly **32×** a single-vector index, and a candidate-generation step that is now a
multi-vector nearest-neighbor problem. This notebook is how that bill is paid.

**PLAID** (the engine behind ColBERTv2) serves late interaction at scale by reusing the two ANN
prerequisites *verbatim*, in three movements:

1. **Representation** — cluster every corpus token into one **shared** centroid set (the IVF coarse
   quantizer) and store each token as a centroid id + a PQ-compressed residual (IVFADC at the token
   level). The shared centroids make the next step cheap, and the residual compression resolves the 32× cost.
2. **The centroid-MaxSim approximation** — approximate $\langle q_i,d_j\rangle$ by $\langle q_i,c(d_j)\rangle$,
   with error *exactly* Cauchy–Schwarz, $|\langle q_i,r_j\rangle|\le\lVert q_i\rVert\lVert r_j\rVert$ — the
   cheap signal used to prune. (The honest gap: this bounds **scores**, not the recall ordering.)
3. **The cascade** — probe → centroid-MaxSim prune → residual decompress + full MaxSim rerank on
   survivors. The one exact statement is the **collapse anchor**: probe everything, prune nothing, rerank
   exactly, and the cascade is brute-force MaxSim to floating point.

This notebook imports the tested reference module (which itself imports the late-interaction, IVF, PQ,
and vMF prerequisites and never reimplements them) and runs every `test_*` claim, so each result below
is verified, not asserted in prose. The `.py` owns every number.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks" / "multi-vector-ann-retrieval"))
import numpy as np
import multi_vector_ann_retrieval as mva

# A query (3 token vectors) vs a document (4 token vectors), and the cheap centroid substitution:
Q = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0]])
D = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0], [0.0, -1.0]])
print("full MaxSim:", round(mva.maxsim_score(Q, D), 3), "(imported from the late-interaction topic)")

## Movement 1 — representation: token IVFADC and the shared-centroid trick

Every token embedding in the corpus goes into **one** k-means — the IVF coarse quantizer — but at token
granularity and *shared across documents*. Each token is then stored as its nearest centroid id plus a
product-quantized residual: literally IVFADC (the IVF + PQ topics) applied to tokens. ColBERT
normalizes tokens to unit norm, so $\lVert a-b\rVert^2=2-2\langle a,b\rangle$ and L2 k-means is cosine
clustering at the objective level — the right quantizer for the job.

The payoff is the **storage collapse**: per token, a centroid id ($\lceil\log_2 K\rceil$ bits) plus a PQ
code ($m\log_2 k^\*$ bits) replaces $d\cdot 32$ raw float bits. At ColBERT scale the 32× raw
multi-vector index collapses to roughly a single-vector index.

In [ ]:
mva.test_spherical_kmeans_equivalence()
mva.test_storage_collapse()

## Movement 2 — the centroid-MaxSim approximation and its Cauchy–Schwarz bound

Because tokens share centroids, $\langle q_i,c(d_j)\rangle$ depends on the document token only through
*which centroid* it landed in, so the query-token-to-centroid score table is computed once and reused
across the whole corpus. The approximation error is **exactly** Cauchy–Schwarz:

$$\langle q_i,d_j\rangle-\langle q_i,c(d_j)\rangle=\langle q_i,r_j\rangle,\qquad |\langle q_i,r_j\rangle|\le\lVert q_i\rVert\,\lVert r_j\rVert,$$

with $r_j=d_j-c(d_j)$ the residual. The max over $j$ is 1-Lipschitz in sup-norm, so the document-level
error is bounded by $\sum_i\lVert q_i\rVert\max_j\lVert r_j\rVert$ — the residual norm is the k-means
distortion inherited from IVF. **But** this bounds scores, not the ranking: a uniform additive bound
does not preserve the top-k set, which is precisely why the cascade still reranks (the next movement).

We first check the synthetic token cloud actually separates by topic (the vMF-$\kappa$ gotcha).

In [ ]:
mva.test_token_cloud_separates()
mva.test_centroid_maxsim_collapses_at_zero_residual()
mva.test_cauchy_schwarz_bound()
mva.test_centroid_maxsim_approximates_full()

## Movement 3 — the cascade, and the collapse anchor

Three stages, each a coarsening of the one below: **Stage 1** generates candidates by probing each query
token's nearest centroid lists (the IVF probe at token level); **Stage 2** prunes by the cheap
centroid-MaxSim score; **Stage 3** decompresses residuals and computes full MaxSim (the imported
`maxsim_score`) only on the survivors.

The **collapse anchor** is the one exact statement: at `nprobe=nlist` (probe every cell), `keep=N`
(prune nothing), and exact-token rerank, the cascade equals brute-force MaxSim to floating point — recall
1.0, identical ordering. Everything above it is a heuristic speed-for-recall trade we measure on a
frontier: PLAID reaches a high recall at a fraction of the brute cost, where centroid-only (no rerank)
plateaus. The frontier is one synthetic cloud, not a universal ranking.

In [ ]:
mva.test_cascade_collapses_to_brute()
mva.test_full_probe_no_prune_is_exact()
mva.test_recall_monotone_in_keep()
mva.test_plaid_beats_brute_at_equal_recall()
mva.test_plaid_vs_centroid_on_one_cloud()

## Finance case study and the viz constants

PLAID is what makes a production late-interaction retriever over filings and transcripts deployable: the
shared centroids and compressed residuals shrink the index, the centroid-MaxSim prune throws out most
filings cheaply, and full MaxSim reranks the survivors. `viz_constants()` prints every number the
interactive laboratory mirrors to the decimal (synthetic vMF tokens, not a trained ColBERT).

In [ ]:
mva.finance_demo()
print()
mva.viz_constants()